# B2 · A single GridSearchCV to choose between different models

**Type**: Implement with scikit-learn · **Level**: ●●○ independent · **Time**: about 45 minutes · **core path ★**

## Objective

Use a `Pipeline` with a placeholder step and a **list of dictionaries** grid to search at the same time between two model families
(logistic regression and random forest) with their own hyperparameters, and read `cv_results_` critically.

## Dataset

`ds.titanic()` with the same split and the same `ColumnTransformer` as exercise B1 (already defined as `prep` in the setup).

## Educational purpose

Learn the advanced `GridSearchCV` syntax: a replaceable step (`clf`), parameters specific to each model, and the computation
of the number of trainings. Understand how significant a score difference is compared to the variability across folds.

## Task

1. Build `pipe = Pipeline([("prep", prep), ("clf", LogisticRegression())])`.
2. Build `grid` as a list of two dictionaries: (1) `clf: [LogisticRegression(max_iter=2000)]` with `clf__C: [0.01, 0.1, 1, 10]`; (2) `clf: [RandomForestClassifier(random_state=0)]` with `clf__max_depth: [3, 6, None]` and `clf__min_samples_leaf: [1, 5]`.
3. Run `GridSearchCV(pipe, grid, cv=cv, scoring="roc_auc")` on the training set and save `search`, `n_candidates` (number of combinations), `n_trainings` (candidates × folds + 1), `best_family` (`"LogisticRegression"` or `"RandomForestClassifier"`) and `test_auc`.
4. From `cv_results_` build `r` (DataFrame) and compute `gap`: the difference in `mean_test_score` between the first and the second ranked, and `best_std`: the `std_test_score` of the best one. Assign to `significant_difference` `True` if `gap > best_std`.

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

df = ds.titanic()
X, y = df.drop(columns="survived"), df["survived"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)
NUM = ["age", "siblings_spouses", "parents_children", "fare"]
CAT = ["sex", "port", "deck", "title"]
cv = StratifiedKFold(5, shuffle=True, random_state=0)

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline

prep = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), NUM),
    ("cat", make_pipeline(SimpleImputer(strategy="most_frequent"), OneHotEncoder(handle_unknown="ignore")), CAT),
    ("pclass", "passthrough", ["pclass"]),
], sparse_threshold=0)

## Your work

In [ ]:
pipe = ...
grid = [
    {...},
    {...},
]

In [ ]:
search = ...
n_candidates = ...
n_trainings = ...
best_family = ...
test_auc = ...

In [ ]:
r = pd.DataFrame(search.cv_results_).sort_values("rank_test_score")
gap = ...
best_std = ...
significant_difference = ...

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='Grid: `{"clf": [LogisticRegression(max_iter=2000)], "clf__C": [0.01, 0.1, 1, 10]}` and `{"clf": [RandomForestClassifier(random_state=0)], "clf__max_depth": [3, 6, None], "clf__min_samples_leaf": [1, 5]}`. `n_candidates = len(search.cv_results_["params"])`.'):
    check(isinstance(grid, list) and len(grid) == 2, "grid = list of two dictionaries", "one dictionary per model family")
    check(n_candidates == 4 + 3 * 2 and n_trainings == 10 * 5 + 1, "10 candidates (4 + 6) and 51 trainings", "candidates x folds + refit")
    check(best_family in ("LogisticRegression", "RandomForestClassifier") and best_family == type(search.best_estimator_[-1]).__name__,
          "best_family consistent with best_estimator_", "use type(search.best_estimator_[-1]).__name__")
    check(0.80 < test_auc < 0.93, "reasonable test AUC", "test_auc = roc_auc_score(y_te, search.predict_proba(X_te)[:, 1])")
    check(close(gap, r["mean_test_score"].iloc[0] - r["mean_test_score"].iloc[1]) and close(best_std, r["std_test_score"].iloc[0]),
          "gap and standard deviation of the best are correct", "first two rows of r, sorted by rank")
    check(significant_difference == (gap > best_std), "judgment consistent with the numbers", "significant_difference = gap > best_std")
    check(gap < 0.02, "the gap between the first two is small (less than 0.02): the choice is between nearly equivalent candidates",
          "look at mean_test_score of the first two")
done()

---
The commented solution is in `solutions/B_with_sklearn/B2_gridsearch_over_several_models.ipynb`. Open it after you pass the check, or after you have really tried.